In [1]:
import os
import glob
import numpy as np
import pandas as pd

def process_single_stock(file_path):

    stock_name = os.path.splitext(os.path.basename(file_path))[0]

    print(f"Processing: {os.path.basename(file_path)}")
    
    df = pd.read_csv(file_path, sep=None, engine='python')
    df.columns = [col.strip() for col in df.columns]
    
    # Check if essential columns exist
    if 'date' not in df.columns or 'close' not in df.columns:
        print(f"Skipping {os.path.basename(file_path)}: Missing required columns.")
        return None, None, None
        
    # df['timestamp'] = pd.to_datetime(df['date'], format='%d-%m-%Y %H:%M')
    df['timestamp'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')
    df['pure_date'] = df['timestamp'].dt.date
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    unique_days = df['pure_date'].unique()
    
    all_days_intraday = []
    all_days_overnight = []
    all_metadata =[]
    previous_day_close = None
    
    for i, day in enumerate(unique_days):
        day_df = df[df['pure_date'] == day]
        
        # --- CRITICAL FIX: SANITY CHECK FOR ZERO OR NEGATIVE PRICES ---
        # If the data provider has buggy data with price <= 0, drop the entire day
        if (day_df['close'] <= 0).any() or ('open' in day_df.columns and (day_df['open'] <= 0).any()):
            # Moving on to next day, but remember to preserve last known valid close if possible
            if len(day_df) > 0 and day_df['close'].iloc[-1] > 0:
                previous_day_close = day_df['close'].iloc[-1]
            continue
            
        start_time = pd.Timestamp(f"{day} 09:15:00")
        expected_timestamps = pd.date_range(start=start_time, periods=375, freq='1min')
        template_df = pd.DataFrame({'timestamp': expected_timestamps})
        
        merged_df = pd.merge(template_df, day_df, on='timestamp', how='left')
        
        # Forward-fill missing elements safely
        merged_df['close'] = merged_df['close'].ffill().bfill()
        
        # Secondary check: if the entire day was empty, ffill/bfill will still leave NaNs or zeros
        if merged_df['close'].isna().any() or (merged_df['close'] <= 0).any():
            continue
            
        closes = merged_df['close'].values
        
        # Calculate returns with an added epsilon safety guard to prevent division by zero
        # (Though our prior check mostly eliminates this, epsilon ensures total safety)
        epsilon = 1e-8
        intraday_return = np.log((closes[1:] + epsilon) / (closes[:-1] + epsilon))
        
        if i == 0 or previous_day_close is None or previous_day_close <= 0:
            overnight_return = 0.0
        else:
            today_open = merged_df['open'].iloc[0] if not pd.isna(merged_df['open'].iloc[0]) else closes[0]
            if today_open <= 0:
                today_open = closes[0]
            overnight_return = np.log((today_open + epsilon) / (previous_day_close + epsilon))
            
        previous_day_close = closes[-1]
        
        # --- ABSOLUTE FINAL GUARD AGAINST INF / NAN CRASHES ---
        if np.isnan(intraday_return).any() or np.isinf(intraday_return).any() or np.isnan(overnight_return) or np.isinf(overnight_return):
            continue # Drop this day out of safety
            
        all_days_intraday.append(intraday_return)
        all_days_overnight.append(overnight_return)

        

        all_metadata.append({"stock": stock_name,"date": str(day)})

        
    if len(all_days_intraday) == 0:
        return None, None, None
        
    stock_intraday_matrix = np.stack(all_days_intraday, axis=0)
    stock_overnight_matrix = np.array(all_days_overnight)
    
    return stock_intraday_matrix, stock_overnight_matrix, all_metadata


def build_and_save_global_dataset(folder_path, output_dir="processed_data"):
    search_path = os.path.join(folder_path, "*.csv")
    all_files = glob.glob(search_path)
    
    global_intraday = []
    global_overnight = []
    global_metadata = []
    
    for file in all_files:

        intra, over, metadata = process_single_stock(file)
        if intra is not None:
            global_intraday.append(intra)
            global_overnight.append(over)
            global_metadata.extend(metadata)
            
    if not global_intraday:
        print("Error: No valid data processed across any files.")
        return
        
    final_intraday = np.concatenate(global_intraday, axis=0)
    final_overnight = np.concatenate(global_overnight, axis=0)
    final_intraday = np.expand_dims(final_intraday, axis=1) # Shape: (Total Profiles, 1, 374)

    assert len(global_metadata) == final_intraday.shape[0]
    assert len(global_metadata) == final_overnight.shape[0]

    metadata_df = pd.DataFrame(global_metadata)

    metadata_df.to_csv(os.path.join(output_dir, "metadata.csv"),index=False)

    print(f"Saved metadata.csv with {len(metadata_df)} records.")

    return final_intraday, final_overnight
    
    # Create output directory if it doesn't exist
    
# --- TO RUN THE PIPELINE ---
output_dir="processed_data"

os.makedirs(output_dir, exist_ok=True)

final_intraday, final_overnight= build_and_save_global_dataset("Dataset_100")




    
# Save arrays to disk
np.save(os.path.join(output_dir, "X_intraday.npy"), final_intraday)
np.save(os.path.join(output_dir, "X_overnight.npy"), final_overnight)

print("\n--- SANITIZATION & SAVING COMPLETE ---")
print(f"Saved to '{output_dir}/'")
print(f"Final Clean Intraday Shape: {final_intraday.shape} (Should contain ZERO NaNs/infs)")
print(f"Final Clean Overnight Shape: {final_overnight.shape}")

# Double check array integrity
assert not np.isnan(final_intraday).any(), "Fatal: Intraday still contains NaNs!"
assert not np.isinf(final_intraday).any(), "Fatal: Intraday still contains Infs!"
print("Integrity verification passed! Data is safe for PyTorch.")


Processing: ABB_minute.csv
Processing: ADANIENSOL_minute.csv
Processing: ADANIENT_minute.csv
Processing: ADANIGREEN_minute.csv
Processing: ADANIPORTS_minute.csv
Processing: ADANIPOWER_minute.csv
Processing: AMBUJACEM_minute.csv
Processing: APOLLOHOSP_minute.csv
Processing: ASIANPAINT_minute.csv
Processing: ATGL_minute.csv
Processing: AXISBANK_minute.csv
Processing: BAJAJ-AUTO_minute.csv
Processing: BAJAJFINSV_minute.csv
Processing: BAJAJHFL_minute.csv
Processing: BAJAJHLDNG_minute.csv
Processing: BAJFINANCE_minute.csv
Processing: BANKBARODA_minute.csv
Processing: BEL_minute.csv
Processing: BHARTIARTL_minute.csv
Processing: BHEL_minute.csv
Processing: BOSCHLTD_minute.csv
Processing: BPCL_minute.csv
Processing: BRITANNIA_minute.csv
Processing: CANBK_minute.csv
Processing: CGPOWER_minute.csv
Processing: CHOLAFIN_minute.csv
Processing: CIPLA_minute.csv
Processing: COALINDIA_minute.csv
Processing: CUMMINSIND_minute.csv
Processing: DABUR_minute.csv
Processing: DIVISLAB_minute.csv
Processing: